# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to Azure AI Foundry Agent Service using the **Microsoft Agent Framework**.
2. Give the agent a **tool** — a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.

## Setup

Before running this notebook, make sure you have:

1. **An Azure AI Foundry project** with a deployed chat model (e.g. `gpt-4o-mini`).
2. **Logged in with the Azure CLI** — run `az login` in your terminal.
3. **Set the required environment variables:**
   - `AZURE_AI_PROJECT_ENDPOINT` — your Azure AI Foundry project endpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.

The cell below installs the Python packages you need.

In [20]:
%pip install agent-framework azure-ai-projects azure-identity -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from dotenv import load_dotenv

# Notebook is at: 01-intro-to-ai-agents/code_samples/ — go up 2 levels to reach .env
_env_path = os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env")
load_dotenv(dotenv_path=_env_path, override=True)

# Map to the env var name the framework expects
os.environ["FOUNDRY_PROJECT_ENDPOINT"] = os.getenv("AZURE_AI_PROJECT_ENDPOINT", "")

print("Endpoint loaded:", bool(os.environ.get("FOUNDRY_PROJECT_ENDPOINT")))

from agent_framework import tool
from agent_framework.foundry import FoundryAgent
from azure.identity import AzureCliCredential

credential = AzureCliCredential()


Endpoint loaded: True


## Creating Your First Agent

An agent needs two things:

- **Instructions** that tell it *who it is* and *how to behave* (a system prompt).
- **Tools** — Python functions decorated with `@tool` that the agent can call to retrieve information or perform actions.

Below we define a simple tool that returns a list of popular vacation destinations. The agent will use this tool when a user asks for travel recommendations.

In [22]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [23]:
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient

client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME"),
    credential=credential,
)

agent = Agent(
    client=client,
    tools=[get_destinations],
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)


Based on the warm beach vibe, my top picks from the list are:

- **Bali** — warm tropical weather, beautiful beaches, great for relaxing and also activities (surfing, temples, day trips).
- **Rio de Janeiro** — warm and lively beach culture (Copacabana/Ipanema) with amazing scenery and nightlife.
- **Sydney** — generally warm in the southern-hemisphere summer, with great beaches and a very comfortable coastal city feel.
- **Cape Town** — can be less consistently “tropical,” but it’s a fantastic coastal destination with stunning beaches (best if you don’t need year-round beach heat).

If you tell me **when you’re going** (month) and whether you want **more relaxing vs. more activities**, I can narrow it to the best match.


## Streaming Responses

For a more interactive experience you can **stream** the agent's response. Instead of waiting for the full reply, the agent yields text chunks as they are generated. This is especially useful in chat interfaces where you want to display output in real time.

In [24]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)


Tokyo is an exciting, high-energy city that blends ultra-modern life with deep traditions—great for travelers who like food, neighborhoods to explore, and lots of things to do.

### Why people love Tokyo
- **Vast variety of experiences:** From futuristic districts (like Shibuya) to historic areas (like Asakusa and Senso-ji Temple).
- **Food heaven:** Sushi, ramen, tempura, izakaya (Japanese pubs), and incredible street food/snacks. You can eat well in almost every budget range.
- **Neighborhood “hopping”:** Tokyo is best enjoyed by exploring different areas—each feels distinct.
- **Efficient public transit:** The subway/train system makes it easy to get around.

### Top things to do
- **Senso-ji Temple (Asakusa):** One of Tokyo’s most iconic traditional sites.
- **Meiji Jingu (Harajuku area):** A peaceful shrine surrounded by forest—great contrast to the city bustle.
- **Shibuya Crossing + Hachiko:** Classic Tokyo moment; lively shopping and nightlife nearby.
- **Tsukiji Outer Market:*

## Summary

In this lesson you learned how to:

- **Create a provider** that connects to Azure AI Foundry Agent Service via `AzureAIProjectAgentProvider`.
- **Define a tool** using the `@tool` decorator so the agent can call your Python functions.
- **Run the agent** with a user message and print its response.
- **Stream responses** for real-time output.

In the next lesson we will explore agentic frameworks in more depth and learn how to give agents more powerful tools and multi-step reasoning capabilities.